In [ ]:
from pyhydra.utils import interactive_map
from pyhydra.data_sources.rainfall import download_era5
import os
from pathlib import Path

# Portable repo-root / data-dir resolution (local clone, Docker, Azure ML)
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))


# ERA5 — Reanalysis Data by Copernicus / ECMWF

## What is ERA5?

[ERA5](https://www.ecmwf.int/en/forecasts/datasets/reanalysis-datasets/era5) is the fifth-generation
**global reanalysis** produced by ECMWF under the Copernicus Climate Change Service (C3S).
A reanalysis **fuses historical observations with a numerical weather model** to produce a
consistent, gap-free global record at every grid point — even where no observations exist.

| Property | Value |
|----------|-------|
| Institution | ECMWF (European Centre for Medium-Range Weather Forecasts) |
| Spatial resolution | ~0.25° (~25 km) globally |
| Temporal resolution | Hourly |
| Coverage | 1940 – present (~5-day delay) |
| Access | Copernicus CDS (free account required) |

## 🆚 ERA5 vs station data vs satellite — when to use which?

| Source | Coverage | Accuracy | Best for |
|--------|----------|----------|----------|
| **ERA5** | Global, 100% | Moderate (reanalysis bias) | Ungauged areas, long consistent records, temperature |
| **AEMET / Meteostat** | Station network | High (direct measurement) | Validation, Spain-specific studies |
| **GPM IMERG** | 60°S–60°N | Moderate-high (satellite+gauge) | Sub-daily precipitation, developing regions |
| **PERSIANN-CCS** | 60°S–60°N | Moderate (IR-only) | Near-real-time, high spatial resolution |

> **Recommendation for Spain:** ERA5 is excellent for temperature and wind.
> For precipitation, validate against AEMET before use — ERA5 underestimates
> extreme rainfall intensity (convective events, orographic enhancement).

## 📦 Available datasets

| Dataset | Resolution | Description |
|---------|-----------|-------------|
| **ERA5 Single Levels** | Hourly / Monthly | Surface variables (precip, temperature, wind, radiation) |
| **ERA5 Land** | Hourly (~9 km) | Land-focused variables at higher resolution |
| **ERA5 Pressure Levels** | Hourly | Atmospheric variables at multiple pressure levels |

> ⚠️ The current code is prepared for **ERA5 Single Levels** (hourly and monthly means).
> ERA5 Land and pressure levels require adapting the dataset endpoint.

## 🌤 Key variables (Single Levels)

| CDS name | Description | Unit | Notes |
|----------|-------------|------|-------|
| `total_precipitation` | Accumulated precip per step | m | × 1000 → mm |
| `2m_temperature` | Air temperature at 2 m | K | − 273.15 → °C |
| `surface_runoff` | Runoff leaving the surface | m | useful for water balance |
| `10m_u_component_of_wind` | U-component wind at 10 m | m/s | |
| `10m_v_component_of_wind` | V-component wind at 10 m | m/s | |

## 🔑 Credentials setup

1. Register at https://cds.climate.copernicus.eu (free)
2. Copy your Personal Access Token from your profile
3. Set it as an environment variable: `export CDS_API_KEY=<your-token>`
   — or create `~/.cdsapirc`:
   ```ini
   url: https://cds.climate.copernicus.eu/api
   key: <your-token>
   ```

## 🔗 References
- [ERA5 documentation](https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation)
- [CDS variable list](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-single-levels)


## 🗺️ Select area of interest

Draw a **rectangle** on the map to define the download region.
The bounding box is read as `coordinates_list[0]` = `[N, W, S, E]` in the next cell.

> **Tip on area size vs download time:**
> - Iberian Peninsula (~10°×9° box): ~25 MB/month at hourly resolution per variable
> - Reducing to a catchment-scale box (e.g., 1°×1°) cuts file size ~100×
> - Use monthly means for trend analysis (12× smaller than hourly)


In [ ]:
coordinates_list = interactive_map(zoom=4, center=(40, -5))

---
## ⚙️ Download configuration

| Parameter | Description |
|-----------|-------------|
| `variables_list` | CDS long names (e.g. `'total_precipitation'`, `'2m_temperature'`) |
| `year_ini` / `year_end` | Year range — ERA5 starts from 1940 |
| `frequency` | `'hourly'` or `'monthly'` — monthly is 12× smaller |
| `max_workers` | Parallel CDS requests (max ~5 to avoid rate limiting) |
| `file_format` | `'netcdf'` recommended; `'grib'` also available |

**Execution modes (controlled by environment variables):**
- `HYDRA_RUN_DOWNLOADS=1` — full download (may take hours for multi-year, multi-variable)
- `HYDRA_DOWNLOAD_SMOKE=1` — quick 1-month test to verify credentials and API path
- Neither set — skips download (useful in CI/build environments)


In [ ]:
# === CONFIGURATION ===
# Credentials are read from the environment. Do not commit personal API keys.
url = os.getenv('CDS_API_URL', 'https://cds.climate.copernicus.eu/api')
api_key = os.getenv('CDS_API_KEY') or os.getenv('CDSAPI_KEY')

# Release/build mode executes notebooks without large external downloads.
# Set HYDRA_DOWNLOAD_SMOKE=1 for a tiny API/download check.
# Set HYDRA_RUN_DOWNLOADS=1 for the full historical download.
RUN_DOWNLOADS = os.getenv('HYDRA_RUN_DOWNLOADS', '0') == '1'
RUN_SMOKE = os.getenv('HYDRA_DOWNLOAD_SMOKE', '0') == '1'

area = coordinates_list[0] if coordinates_list else [44, -10, 35, 5]  # [N, W, S, E]
download_base_dir = os.getenv('HYDRA_ERA5_DIR', str(DATA_DIR / 'era5'))
os.makedirs(download_base_dir, exist_ok=True)

variables_list = ['total_precipitation', 'surface_runoff', '2m_temperature']
year_ini = 1990
year_end = 2020

print(f'ERA5 output dir : {download_base_dir}')
print(f'Run full download: {RUN_DOWNLOADS}')
print(f'Run smoke check  : {RUN_SMOKE}')
print(f'API key present  : {bool(api_key)}')


In [ ]:
if RUN_DOWNLOADS or RUN_SMOKE:
    if not api_key:
        print('CDS API key not configured. Set CDS_API_KEY or CDSAPI_KEY to run downloads.')
    elif RUN_SMOKE:
        smoke_dir = str(Path(download_base_dir) / 'smoke')
        smoke_area = [43.35, -4.15, 43.15, -3.95]  # small box around Los Corrales de Buelna
        print('Running ERA5 smoke download: 1 variable, 1 month, monthly means, small area...')
        download_era5(
            folder_out=smoke_dir,
            api_key=api_key,
            url=url,
            area=smoke_area,
            variables_list=['total_precipitation'],
            years=[2020],
            months=[1],
            max_workers=1,
            file_format='netcdf',
            frequency='monthly',
        )
        print(f'ERA5 smoke output -> {smoke_dir}')
    else:
        print('Running full ERA5 download. This is intentionally disabled during image builds.')
        download_era5(
            folder_out=download_base_dir,
            api_key=api_key,
            url=url,
            area=area,
            variables_list=variables_list,
            years=range(year_ini, year_end + 1),
            months=range(1, 13),
            max_workers=5,
            file_format='netcdf',
            frequency='hourly',
        )
else:
    print('Skipping ERA5 download in release mode. Set HYDRA_DOWNLOAD_SMOKE=1 for a tiny API test or HYDRA_RUN_DOWNLOADS=1 for the full download.')

Running ERA5 smoke download: 1 variable, 1 month, monthly means, small area...
Saved: /tmp/hydra_era5_smoke/smoke/ERA5_monthly_2020_01_combined.nc (elapsed: 0:00:26.222410)
ERA5 smoke output -> /tmp/hydra_era5_smoke/smoke


2026-06-13 12:01:27,568 INFO [2026-06-11T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 15 June. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure-part-2/150414).
2026-06-13 12:01:28,229 INFO Request ID is 749885b5-47b1-4987-bd9d-747c3a617d55
2026-06-13 12:01:30,316 INFO status has been updated to accepted
2026-06-13 12:01:44,089 INFO status has been updated to running
2026-06-13 12:01:51,820 INFO status has been updated to successful


---
## 📊 Post-download: loading and using ERA5 data

ERA5 downloads as NetCDF files (one per variable per month or year). Load with xarray:

```python
import xarray as xr
import glob

# Load all monthly precipitation files
files = sorted(glob.glob('/workspace/data/era5/ERA5_monthly_*_total_precipitation*.nc'))
ds = xr.open_mfdataset(files, combine='by_coords')

# Extract point series at a specific location
LAT, LON = 43.26, -4.05   # Los Corrales de Buelna
pt = ds['tp'].sel(latitude=LAT, longitude=LON, method='nearest')
precip_mm = pt.to_dataframe()['tp'] * 1000  # m → mm
```

#### ⚠️ Known biases to account for

- **Precipitation**: ERA5 underestimates extreme intensity; overestimates drizzle frequency
- **Mountain regions**: orographic enhancement is smoothed at 25 km resolution
- **Temperature**: generally reliable, but cold biases in valley floors
- **Runoff**: `surface_runoff` in ERA5 is from the HTESSEL land-surface model —
  it should not be used directly as catchment runoff without calibration

Always validate against AEMET / Meteostat observations before using ERA5 as model forcing.
See the `bias_correction` notebook for QDM correction.
